In [1]:
import json
from pathlib import Path
PROJECT_ROOT = next(
    p for p in Path.cwd().parents if p.name == "LaboroTomato"
)

In [2]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

import matplotlib.pyplot as plt

In [3]:
def build_model(name: str, num_classes: int) -> nn.Module:
    name = name.lower()
    if name == "mobilenet_v2":
        weights = models.MobileNet_V2_Weights.IMAGENET1K_V1
        m = models.mobilenet_v2(weights=weights)
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, num_classes)
        return m
    if name == "efficientnet_v2_s":
        weights = models.EfficientNet_V2_S_Weights.IMAGENET1K_V1
        m = models.efficientnet_v2_s(weights=weights)
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, num_classes)
        return m
    raise ValueError(f"Unknown model: {name}")

In [4]:
def softmax_np(logits: np.ndarray) -> np.ndarray:
    # stable softmax
    z = logits - logits.max(axis=1, keepdims=True)
    exp = np.exp(z)
    return exp / exp.sum(axis=1, keepdims=True)

In [5]:
def compute_ece(conf: np.ndarray, correct: np.ndarray, n_bins: int = 15):
    """
    ECE = sum_k (|acc_k - conf_k| * (n_k / N))
    where conf_k is avg confidence in bin k and acc_k is avg accuracy in bin k.
    """
    assert conf.shape == correct.shape
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    bin_ids = np.digitize(conf, bins[1:-1], right=False)  # 0..n_bins-1

    N = len(conf)
    ece = 0.0
    bin_stats = []

    for b in range(n_bins):
        mask = bin_ids == b
        n_b = int(mask.sum())
        if n_b == 0:
            bin_stats.append({
                "bin": b,
                "count": 0,
                "conf_avg": None,
                "acc_avg": None,
                "bin_low": float(bins[b]),
                "bin_high": float(bins[b+1]),
            })
            continue

        conf_avg = float(conf[mask].mean())
        acc_avg = float(correct[mask].mean())
        ece += abs(acc_avg - conf_avg) * (n_b / N)

        bin_stats.append({
            "bin": b,
            "count": n_b,
            "conf_avg": conf_avg,
            "acc_avg": acc_avg,
            "bin_low": float(bins[b]),
            "bin_high": float(bins[b+1]),
        })

    return float(ece), bin_stats

In [6]:
def plot_reliability(bin_stats, out_path: Path):
    # reliability diagram: x=confidence, y=accuracy
    confs = []
    accs = []
    counts = []

    for s in bin_stats:
        if s["count"] > 0:
            confs.append(s["conf_avg"])
            accs.append(s["acc_avg"])
            counts.append(s["count"])

    fig = plt.figure(figsize=(6, 6))
    # main curve
    plt.plot(confs, accs, marker="o", linewidth=1)
    # perfect calibration line
    plt.plot([0, 1], [0, 1], linestyle="--", linewidth=1)
    plt.xlim(0, 1)
    plt.ylim(0, 1)
    plt.xlabel("Confidence (avg per bin)")
    plt.ylabel("Accuracy (avg per bin)")
    plt.title("Reliability Diagram")
    plt.tight_layout()
    fig.savefig(out_path, dpi=200)
    plt.close(fig)

In [7]:
def plot_confidence_hist(conf_correct: np.ndarray, conf_wrong: np.ndarray, out_path: Path):
    fig = plt.figure(figsize=(7, 4))
    plt.hist(conf_correct, bins=20, alpha=0.7, label="Correct")
    plt.hist(conf_wrong, bins=20, alpha=0.7, label="Wrong")
    plt.xlabel("Max softmax confidence")
    plt.ylabel("Count")
    plt.title("Confidence distribution: correct vs wrong")
    plt.legend()
    plt.tight_layout()
    fig.savefig(out_path, dpi=200)
    plt.close(fig)

In [8]:
def main():
    RUN_DIR = PROJECT_ROOT / "runs_cpu_local" / "baseline_mobilenetv2" / "20251228-003415"
    DATA_ROOT = PROJECT_ROOT / "dataset" / "cropped"

    BEST_CKPT = RUN_DIR / "model_best.pt"
    CLASS_MAP = RUN_DIR / "class_to_idx.json"

    assert BEST_CKPT.exists(), f"Missing checkpoint: {BEST_CKPT}"
    assert CLASS_MAP.exists(), f"Missing class map: {CLASS_MAP}"
    assert (
        DATA_ROOT / "test").exists(), f"Missing test folder: {DATA_ROOT / 'test'}"

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print("Device:", device)

    # Load checkpoint/config
    ckpt = torch.load(BEST_CKPT, map_location="cpu")
    cfg = ckpt.get("config", {})
    model_name = cfg.get("model_name", "mobilenet_v2")
    image_size = int(cfg.get("image_size", 224))
    batch_size = int(cfg.get("batch_size", 64))

    with open(CLASS_MAP, "r", encoding="utf-8") as f:
        class_to_idx = json.load(f)

    idx_to_class = {v: k for k, v in class_to_idx.items()}
    class_names = [idx_to_class[i] for i in range(len(idx_to_class))]
    num_classes = len(class_names)

    print("Model:", model_name)
    print("Classes:", class_names)

    # Transforms
    imagenet_mean = (0.485, 0.456, 0.406)
    imagenet_std = (0.229, 0.224, 0.225)

    eval_tfms = transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(),
        transforms.Normalize(imagenet_mean, imagenet_std),
    ])

    # Dataset (keep file paths)
    test_ds = datasets.ImageFolder(
        root=str(DATA_ROOT / "test"), transform=eval_tfms)
    test_loader = DataLoader(
        test_ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,   # Windows-safe
        pin_memory=True
    )

    # Model
    model = build_model(model_name, num_classes=num_classes)
    model.load_state_dict(ckpt["model_state"])
    model.to(device)
    model.eval()

    # Inference: store logits to compute confidence
    all_logits = []
    all_y = []
    all_paths = []

    with torch.no_grad():
        for x, y in test_loader:
            x = x.to(device, non_blocking=True)
            logits = model(x).cpu().numpy()
            all_logits.append(logits)
            all_y.append(y.numpy())

    logits = np.concatenate(all_logits, axis=0)
    y_true = np.concatenate(all_y, axis=0)

    probs = softmax_np(logits)
    y_pred = probs.argmax(axis=1)
    conf = probs.max(axis=1)
    correct = (y_pred == y_true).astype(np.float32)

    # ECE + bin stats
    n_bins = 15
    ece, bin_stats = compute_ece(conf, correct, n_bins=n_bins)

    # Summary: confidence when correct vs wrong
    conf_correct = conf[correct == 1]
    conf_wrong = conf[correct == 0]

    summary = {
        "n": int(len(conf)),
        "accuracy": float(correct.mean()),
        "ece": float(ece),
        "n_bins": int(n_bins),
        "confidence_correct_mean": float(conf_correct.mean()) if len(conf_correct) else None,
        "confidence_wrong_mean": float(conf_wrong.mean()) if len(conf_wrong) else None,
        "confidence_correct_median": float(np.median(conf_correct)) if len(conf_correct) else None,
        "confidence_wrong_median": float(np.median(conf_wrong)) if len(conf_wrong) else None,
        "confidence_wrong_p90": float(np.percentile(conf_wrong, 90)) if len(conf_wrong) else None,
    }

    print("\n=== Calibration on Test ===")
    print(f"Accuracy: {summary['accuracy']:.4f}")
    print(f"ECE     : {summary['ece']:.4f}  (bins={n_bins})")
    print(
        f"Mean confidence (correct): {summary['confidence_correct_mean']:.4f}")
    print(f"Mean confidence (wrong)  : {summary['confidence_wrong_mean']:.4f}")
    print(f"P90 confidence (wrong)   : {summary['confidence_wrong_p90']:.4f}")

    paths = [p for (p, _) in test_ds.samples]
    pred_df = pd.DataFrame({
        "path": paths,
        "y_true": y_true,
        "y_pred": y_pred,
        "true_label": [class_names[i] for i in y_true],
        "pred_label": [class_names[i] for i in y_pred],
        "confidence": conf,
        "correct": correct.astype(int),
    })
    pred_csv = RUN_DIR / "predictions.csv"
    pred_df.to_csv(pred_csv, index=False)

    # Plots
    rel_path = RUN_DIR / "reliability_diagram.png"
    hist_path = RUN_DIR / "confidence_hist.png"
    plot_reliability(bin_stats, rel_path)
    plot_confidence_hist(conf_correct, conf_wrong, hist_path)

    # Save metrics json
    out = {
        "summary": summary,
        "class_names": class_names,
        "bin_stats": bin_stats,
        "checkpoint": str(BEST_CKPT),
        "data_root": str(DATA_ROOT),
        "split": "test",
    }
    out_path = RUN_DIR / "metrics.json"
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(out, f, indent=2)

    print(f"\nSaved: {out_path}")
    print(f"Saved: {rel_path}")
    print(f"Saved: {hist_path}")
    print(f"Saved: {pred_csv}")



In [9]:
if __name__ == "__main__":
    main()

Device: cpu
Model: mobilenet_v2
Classes: ['fully_ripened', 'green', 'half_ripened']

=== Calibration on Test ===
Accuracy: 0.8197
ECE     : 0.1125  (bins=15)
Mean confidence (correct): 0.9451
Mean confidence (wrong)  : 0.8556
P90 confidence (wrong)   : 0.9999

Saved: c:\Users\User\Desktop\LaboroTomato\runs_cpu_local\baseline_mobilenetv2\20251228-003415\metrics.json
Saved: c:\Users\User\Desktop\LaboroTomato\runs_cpu_local\baseline_mobilenetv2\20251228-003415\reliability_diagram.png
Saved: c:\Users\User\Desktop\LaboroTomato\runs_cpu_local\baseline_mobilenetv2\20251228-003415\confidence_hist.png
Saved: c:\Users\User\Desktop\LaboroTomato\runs_cpu_local\baseline_mobilenetv2\20251228-003415\predictions.csv


*Calibration summary (natural background baseline):*\
***Calibration analysis on the test split revealed a persistent mismatch between predicted confidence and empirical accuracy. Using 15 equal-width bins, the baseline MobileNetV2 achieved an Expected Calibration Error (ECE) of 0.112, indicating moderate but systematic miscalibration. Although overall accuracy reached 81.97%, incorrect predictions were frequently assigned high confidence: the mean confidence for misclassified samples was 0.856, with a median of 0.966 and a 90th percentile close to 1.0. The reliability diagram further showed consistent overconfidence in the highest-confidence regime (average confidence ≈ 0.993 versus accuracy ≈ 0.865), demonstrating that the model can remain highly confident even when wrong.***

In [1]:
!jupyter nbconvert --to html 04_eval_calibration.ipynb

[NbConvertApp] Converting notebook 04_eval_calibration.ipynb to html
[NbConvertApp] Writing 323855 bytes to 04_eval_calibration.html
